<img src='./img/up_bann.png' style="display: block; margin: 0 auto;">

*This algorithm has been **community** contributed to APEx and has not undergine scientific validation review* 

# Burnt scar visualization — Siverskyi Donets, east of Luhansk (October 2020)

This notebook runs the `wildfire_visualization` algorithm on a cloud-free **Sentinel**-2 scene from 5 October 2020. The image is true colour, with burnt ground (low Normalised Burn Ratio) highlighted in yellow-orange.

The `wildfire_visualization` UDP (Sentinel-2 L2A true colour with a square-root stretch; pixels with NBR < -0.25 get a +0.3 red/green boost so the burnt scars show as orange-yellow) on CDSE over the late-September 2020 wildfires along the Siverskyi Donets river, then plots the result.

Scenes picked up by the window (openEO end dates are exclusive; the UDP keeps only scenes with ≤ 30% cloud and uses the first one):

| Window | Sentinel-2 L2A scenes (tile cloud cover) | Used |
|---|---|---|
| 2020-09-30 → 2020-10-06 | 2020-09-30 (45–78%), 2020-10-02 (79–93%), 2020-10-05 (1%) | 2020-10-05 |

### Import Required libraries

In [ ]:
from pathlib import Path
import os
from getpass import getpass
from esa_apex_toolbox.algorithms import GithubAlgorithmRepository

import openeo

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.patches import Patch

### Define Costum Functions

The following function, allows us to read our UDP's result:

In [ ]:
def read_rgb(path):
    """Read a 3-band wildfire_visualization GeoTIFF as an (H, W, 3) 0-1 array plus its extent in km."""
    with rasterio.open(path) as src:
        data = src.read()
        b = src.bounds
    # The UDP output is already display-scaled (sqrt stretch); burnt pixels can
    # exceed 1 after the +0.3 boost, so clip to 0-1.
    rgb = np.clip(np.nan_to_num(np.moveaxis(data, 0, -1)), 0, 1)
    extent = [b.left / 1000, b.right / 1000, b.bottom / 1000, b.top / 1000]
    return rgb, extent


### Authentication

To run the jobs directly in the openEO CDSE federation, ir is required to authenticate with our CDSE credentials:

In [ ]:
connection = openeo.connect("https://openeo.dataspace.copernicus.eu/").authenticate_oidc()

Also, to explore the APEx Catalog, a GitHub token is recommended (GitHub normally allows 60 unauthenticated **REST API requests per hour per IP address**):

In [ ]:
github_token = os.environ.get('GITHUB_TOKEN') or os.environ.get('GH_TOKEN')
if not github_token:
    github_token = getpass('GitHub token (optional; Enter for unauthenticated access): ') or None

We define our area of interest, the time frame of interest and our output paths

In [ ]:
# Bounding box and temporal extent
shape = {"west": 39.4750, "south": 48.5626, "east": 39.6472, "north": 48.6704}
start_date, end_date = ["2020-09-30", "2020-10-06"]

# define our output paths
output = Path("wildfire_visualization_luhansk.tiff")

Next step is selecting our algorithm and accessing the `udp_link` trough its `id`. For this case it corresponds to `wildfire_visualization`

In [ ]:
repo = GithubAlgorithmRepository(
    owner='ESA-APEx',
    repo='apex_algorithms',
    folder='algorithm_catalog',
    branch='main',
    token=github_token,
)

process_id = "wildfire_visualization"
wf_algorithm = repo.get_algorithm(process_id)

wf_algorithm

We define our `datacube_from_process` based on our selected algorithm and its `udp_link.href` (hosted on github)

In [ ]:
burnt_scar_luhansk = connection.datacube_from_process(
    process_id,
    namespace=wf_algorithm.udp_link.href,
    time=[start_date, end_date],
    bounding_box=shape,
    cloud_cover=30,
)

## Burnt scar visualization

We define our job characteristics in the CDSE Open EO federation, and monitour our progress.

In [ ]:
job = burnt_scar_luhansk.create_job(
    title=f"Burnt scar visualization for {start_date} to {end_date}",
    out_format="GTiff",
)
job.start_and_wait()

Finally we access our results for visualisation:

In [ ]:
# The result is a single 3-band GeoTIFF. If the download breaks, rerun only
# this cell; the job does not need to be resubmitted.
job.get_results().download_file(output)

## Burnt scars

Based on our `read_rgb` funciton, we can explore our results.

In [ ]:
rgb, extent = read_rgb(output)

fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(rgb, extent=extent)
ax.set_title("Sentinel-2 L2A 2020-10-05", fontsize=13)
ax.set_xlabel("Easting (km, UTM 37N)")
ax.set_ylabel("Northing (km, UTM 37N)")

fig.legend(
    handles=[
        Patch(color="#e0a040", label="Burnt scar (NBR < -0.25, red/green +0.3)"),
        Patch(color="#5a6a4a", label="Unburnt (Sentinel-2 true colour, sqrt stretch)"),
    ],
    loc="lower center",
    ncol=2,
    frameon=False,
    fontsize=11,
)
fig.suptitle("Burnt scar visualization — Siverskyi Donets, east of Luhansk (October 2020)", fontsize=16)
fig.tight_layout(rect=(0, 0.05, 1, 0.97))
fig.savefig("wildfire_visualization_luhansk.png", dpi=200)
plt.show()

The figure shows a large burn scars along the river, with smoke plumes from fires that were still active.<br>

It shows how much land burned near the front line, which is useful for damage assessment and planning humanitarian help. Some harvested fields in the corners are also highlighted, so don't count every highlighted patch as burnt.

<img src='./img/low_bann.png' style="display: block; margin: 0 auto;">